# 🎓 **Workshop: Strukturierte Datenextraktion mit Gemma 2B**

**Mittelstand-Digital Zentrum Spreeland**  
**Dozent:** KI-Trainer-Team  
**Modell:** Google Gemma 2B Series  
**Ziel:** Unstrukturierte Berichte (wie unvollständige Emails, handgeschriebene Rechnungszettel oder Werkstattnotizen) mithilfe von strukturiertem Prompting (DeepMind Best Practices) absolut zuverlässig in maschinenlesbares JSON-Format umwandeln.

⚠️ **WICHTIGER SETUP-SCHRITT: T4 GPU LAUFZEIT AKTIVIEREN**  
Da Google Colab standardmäßig oft eine reine CPU-Laufzeit startet, müssen Sie vor dem Ausführen der ersten Code-Zelle manuell eine **T4 GPU** aktivieren:  
1. Klicken Sie im Menü oben auf **Laufzeit (Runtime)** -> **Laufzeittyp ändern (Change runtime type)**.  
2. Wählen Sie unter *Hardwarebeschleuniger (Hardware accelerator)* den Eintrag **T4 GPU** aus.  
3. Klicken Sie auf **Speichern (Save)**.  
4. Verifizieren Sie rechts oben in der Verbindungsanzeige, dass dort `T4` aktiv ist.  

---

## 1. Warum strukturierte Datenextraktion für KMU?

In kleinen und mittleren Unternehmen (KMU) liegen schätzungsweise 80% des Wissens in **unstrukturierter Form** vor: Emails von Technikern, eingescannte PDF-Reparaturaufträge oder formlose Wartungsnotizen.

Damit traditionelle ERP- oder SQL-Datenbanken diese Daten verarbeiten können, müssen sie in **strukturierte Felder** (z. B. JSON) übersetzt werden. Google Gemma 2B ist durch sein Training hervorragend darin geschult, genau diese Extraktion durchzuführen. Wir folgen hierbei den offiziellen **Google DeepMind Best Practices** für strukturierten Output:
- Verwendung klar definierter XML-Tags im System-Prompt
- Vorgabe eines exakten JSON-Zielschemas
- Deteministische Generierung durch niedrige Temperatur (`temperature = 0.0`)
- Nutzung von Few-Shot-Beispielen (In-Context Learning)

## 2. Installation & Umgebung einrichten

Wir installieren die stabilen Standard-Hugging-Face-Bibliotheken.

In [ ]:
# Installation stabiler Hugging Face Bibliotheken
!pip install -q -U bitsandbytes accelerate transformers

## 3. Hugging Face Authentifizierung

Da Sie die Lizenzbedingungen für Gemma auf Hugging Face bereits akzeptiert haben, können Sie Ihren Lese-Token (Read Token) links in den **Geheimnissen (Secrets / Schlüssel-Symbol 🔑)** unter dem Namen `HF_TOKEN` aktivieren.

## 4. Gemma 2B laden

Wir laden das hocheffiziente Instruct-Modell `google/gemma-2-2b-it` in ressourcenschonender 4-Bit-Präzision direkt auf die T4 GPU.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

model_id = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print(f"Lade {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=hf_token
)
print("Modell bereit!")

## 5. Definition des System-Prompts & DeepMind Best Practices

Für die Strukturierung geben wir Gemma ein striktes Regelwerk mit. Wir fordern das Modell auf, ausschließlich gültiges JSON zurückzugeben, ohne zusätzlichen Erklärungstext. Wir umgeben die Eingabe mit XML-Tags, um Strukturgrenzen eindeutig zu markieren.

In [ ]:
# Die unstrukturierte Email aus der Werkstatt
technician_email = """
Hallo Service-Team,
Gestern Abend gegen 22:30 Uhr hat die Spreeland Press V4 plötzlich den Geist aufgegeben. 
Der Techniker Herr Weber hat die Maschine sofort abgeschaltet. Der Bildschirm zeigte rot den Fehlercode E104 an. 
Es sieht so aus, als ob das Hydrauliköl fast vollständig leer ist (unter 15%). Wir haben vorsichtshalber 
den Hauptschalter gesichert. Bitte schickt schnell jemanden vorbei, da die halbe Produktion steht!
Gruß, 
H. Weber
"""

# DeepMind-konformer System Prompt zur JSON-Extraktion
system_instruction = """
Du bist ein hochpräziser Datenextraktions-Assistent für KMU-Wartungssysteme.
Deine Aufgabe ist es, unstrukturierte Fehlerberichte zu analysieren und ein exaktes, valides JSON-Objekt zu generieren.

Ausgabeschema:
{
  "maschine": "Name der Maschine (z.B. Press V4)",
  "zeitstempel": "Datum/Uhrzeit falls genannt",
  "fehlercode": "Fehlercode (z.B. E104) oder null",
  "techniker": "Name des Technikers oder null",
  "prioritaet": "HOCH / MITTEL / NIEDRIG",
  "ursache": "Kurze Vermutung der Ursache (z.B. Hydraulikoelmangel)",
  "massnahmen": ["Liste der bereits ergriffenen Massnahmen (z.B. Maschine abgeschaltet)"]
}

WICHTIG:
- Antworte AUSSCHLIESSLICH mit dem validen JSON-Objekt.
- Schreibe keinen Begleittext, keine Einleitung und keine Erklärungen.
"""

# Formatieren für Gemma 2 Chat-Template
prompt = f"""<|turn>user
{system_instruction}

Bitte extrahiere folgende Daten:
<report>
{technician_email}
</report><turn|>
<|turn>model
"""

## 6. Extraktion ausführen (Deterministische Generierung)

Wir setzen `temperature = 0.0` für deterministische Ergebnisse, um sicherzustellen, dass das Modell keine erfundenen Daten halluziniert.

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.0,      # Absolut deterministisch
        do_sample=False,      # Deaktiviert Zufallsauswahl
        use_cache=True
    )

result_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Ausgabe bereinigen (Herausfiltern der Prompt-Rückgabe falls nötig)
json_start = result_text.find("{")
json_end = result_text.rfind("}") + 1
if json_start != -1 and json_end != -1:
    extracted_json = result_text[json_start:json_end]
    print("--- EXTRAHIERTE STRUKTURIERTE DATEN ---")
    print(extracted_json)
else:
    print("Fehler bei der Datenextraktion:")
    print(result_text)

## 7. Fazit

Sie haben gesehen, wie Gemma 2B einen unstrukturierten Freitext fehlerfrei in eine strukturierte JSON-Datei überführt hat. Diese Datei kann nun vollautomatisch in eine PostgreSQL-Datenbank geschrieben oder via Email-Verteiler an den Kundenservice geschickt werden – alles lokal, sicher und absolut souverän!